# F3-D — LoRA Fine-Tuning + Ensemble Ponderado

**Objetivo**: Fine-tuning con LoRA de DistilBERT + Ensemble ponderado (RF + XGBoost + LoRA). Notebook GPU.

**Tiempo estimado**: ~2h (GPU T4)


## 1. Instalar dependencias


In [ ]:
!pip install -q polars mlflow xgboost transformers -U
!pip install -q peft -U


In [ ]:
import numpy as np
import torch
import gc
import os
import json
import time
import mlflow
import matplotlib.pyplot as plt
import pickle
from google.colab import drive
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer, DataCollatorWithPadding)
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')


## 2. Montar Google Drive y cargar datos


In [ ]:
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/ML/proyecto_integrador"
EMB_DIR = f"{DRIVE_BASE}/embeddings"
REPORTS_DIR = f"{DRIVE_BASE}/reports"
PREDS_DIR = f"{DRIVE_BASE}/preds"
RANDOM_STATE = 42
BATCH_SIZE = 256
MAX_LENGTH = 128

for d in [REPORTS_DIR, PREDS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Cargando embeddings, features y textos desde F3-A...")
# Embeddings
X_train_emb = np.load(f"{EMB_DIR}/train_embeddings.npy")
X_val_emb   = np.load(f"{EMB_DIR}/val_embeddings.npy")
X_test_emb  = np.load(f"{EMB_DIR}/test_embeddings.npy")
# Engineered features
eng_train = np.load(f"{EMB_DIR}/train_eng_features.npy")
eng_val   = np.load(f"{EMB_DIR}/val_eng_features.npy")
eng_test  = np.load(f"{EMB_DIR}/test_eng_features.npy")
# Labels
y_train = np.load(f"{EMB_DIR}/train_labels.npy")
y_val   = np.load(f"{EMB_DIR}/val_labels.npy")
y_test  = np.load(f"{EMB_DIR}/test_labels.npy")
# Textos crudos (para LoRA)
with open(f"{EMB_DIR}/train_texts.pkl", 'rb') as f:
    X_train_texts = pickle.load(f)
with open(f"{EMB_DIR}/val_texts.pkl", 'rb') as f:
    X_val_texts = pickle.load(f)
with open(f"{EMB_DIR}/test_texts.pkl", 'rb') as f:
    X_test_texts = pickle.load(f)

# Concatenar para clasicos
X_train = np.concatenate([X_train_emb, eng_train], axis=1)
X_val   = np.concatenate([X_val_emb, eng_val], axis=1)
X_test  = np.concatenate([X_test_emb, eng_test], axis=1)

print(f"Datos cargados: train {len(X_train_texts)}, val {len(X_val_texts)}, test {len(X_test_texts)}")


## 3. Cargar predicciones de RF y XGBoost (desde F3-B)

Si no existen (porque F3-B no se ejecutó), se cargan modelos pre-entrenados.


In [ ]:
PREDS_DIR = f"{DRIVE_BASE}/preds"

if os.path.exists(f"{PREDS_DIR}/y_pred_rf.npy"):
    print("Cargando predicciones de F3-B...")
    y_pred_rf   = np.load(f"{PREDS_DIR}/y_pred_rf.npy")
    y_pred_xgb  = np.load(f"{PREDS_DIR}/y_pred_xgb.npy")
    with open(f"{PREDS_DIR}/part1_results.json") as f:
        part1_results = json.load(f)
    rf_metrics  = [r for r in part1_results if r['model_name'] == 'Random Forest'][0]
    xgb_metrics = [r for r in part1_results if r['model_name'] == 'XGBoost'][0]
    print(f"RF F1: {rf_metrics['f1_macro']}, XGB F1: {xgb_metrics['f1_macro']}")
else:
    print("Predicciones de F3-B no encontradas. Entrenando modelos aquí...")
    from sklearn.ensemble import RandomForestClassifier
    from xgboost import XGBClassifier
    rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)
    xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                        random_state=RANDOM_STATE, eval_metric='mlogloss')
    xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    y_pred_xgb = xgb.predict(X_test)
    print("Modelos clásicos entrenados localmente")


## 4. LoRA Fine-Tuning


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_dict({'text': X_train_texts, 'label': y_train})
val_ds   = Dataset.from_dict({'text': X_val_texts, 'label': y_val})
test_ds  = Dataset.from_dict({'text': X_test_texts, 'label': y_test})

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds   = val_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {'f1_macro': f1_score(labels, predictions, average='macro'),
            'accuracy': accuracy_score(labels, predictions)}


In [ ]:
model_cls = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3
).to(device)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['q_lin', 'k_lin', 'v_lin', 'out_lin']
)
model_lora = get_peft_model(model_cls, lora_config)
model_lora.print_trainable_parameters()


In [ ]:
lora_args = TrainingArguments(
    output_dir='/content/lora_checkpoints',
    eval_strategy='epoch',
    save_strategy='epoch',
    per_device_train_batch_size=128,
    per_device_eval_batch_size=256,
    num_train_epochs=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    report_to='none',
)

trainer_lora = Trainer(
    model=model_lora,
    args=lora_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Iniciando LoRA fine-tuning...")
start = time.time()
trainer_lora.train()
lora_time = time.time() - start
print(f"LoRA completado en {lora_time:.0f}s")


## 5. Evaluar LoRA


In [ ]:
y_pred_lora = trainer_lora.predict(test_ds).predictions.argmax(-1)
lora_f1 = f1_score(y_test, y_pred_lora, average='macro')
print(f"LoRA test F1-macro: {lora_f1:.4f}")

lora_metrics, _ = None, None
def eval_and_record(name, y_true, y_pred, training_time):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1, 2])
    per_class = {
        label: {'precision': round(p[i], 4), 'recall': round(r[i], 4), 'f1': round(f[i], 4)}
        for i, label in enumerate(['Negativo', 'Neutro', 'Positivo'])
    }
    return {
        'model_name': name,
        'training_time_seconds': round(training_time, 2),
        'f1_macro': round(f1_score(y_true, y_pred, average='macro'), 4),
        'precision_macro': round(precision_score(y_true, y_pred, average='macro'), 4),
        'recall_macro': round(recall_score(y_true, y_pred, average='macro'), 4),
        'accuracy': round(accuracy_score(y_true, y_pred), 4),
        'per_class': per_class,
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
    }

lora_metrics = eval_and_record('DistilBERT + LoRA', y_test, y_pred_lora, lora_time)
results = []
if 'rf_metrics' in dir():
    results.append(rf_metrics)
    results.append(xgb_metrics)
results.append(lora_metrics)


## 6. Learning Curves


In [ ]:
def _plot_learning_curve(log_history, title):
    train_loss = [x['loss'] for x in log_history if 'loss' in x and 'eval_loss' not in x]
    eval_loss = [x['eval_loss'] for x in log_history if 'eval_loss' in x]
    eval_f1 = [x.get('eval_f1_macro', None) for x in log_history if 'eval_loss' in x]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(range(1, len(train_loss) + 1), train_loss, label='Train loss', color='#3498db', linewidth=2)
    epochs = list(range(1, len(eval_loss) + 1))
    ax1.plot(epochs, eval_loss, label='Val loss', color='#e74c3c', linewidth=2, marker='o')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title(f'{title} - Loss')
    ax2.plot(epochs, eval_f1, label='Val F1', color='#2ecc71', linewidth=2, marker='o')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('F1-macro'); ax2.legend(); ax2.set_title(f'{title} - Val F1')
    plt.tight_layout(); plt.show()

_plot_learning_curve(trainer_lora.state.log_history, 'LoRA')


## 7. Ensemble Ponderado


In [ ]:
print("\n" + "="*60)
print("Ensemble ponderado")
print("="*60)

model_preds = {
    'Random Forest': y_pred_rf,
    'XGBoost': y_pred_xgb,
    'DistilBERT + LoRA': y_pred_lora,
}

weights_list = []
preds_list = []
for r in results:
    if r['model_name'] in model_preds:
        weights_list.append(r['f1_macro'])
        preds_list.append(model_preds[r['model_name']])

weights_list = np.array(weights_list)
preds_list = np.array(preds_list)

# Weighted majority vote: sumar votos ponderados por clase
n_classes = 3
weighted_votes = np.zeros((len(y_test), n_classes))
for w, pred in zip(weights_list, preds_list):
    for c in range(n_classes):
        weighted_votes[:, c] += w * (pred == c)

y_pred_ensemble = np.argmax(weighted_votes, axis=1)

weights_str = ', '.join([f"{r['model_name']}: {w:.4f}" for r, w in zip(results, weights_list)])
print(f"Pesos del ensemble: {weights_str}")

ens_metrics = eval_and_record('Ensemble (ponderado)', y_test, y_pred_ensemble, 0)
results.append(ens_metrics)
print(f"Ensemble test F1-macro: {ens_metrics['f1_macro']}")


## 8. Guardar predicciones


In [ ]:
os.makedirs(PREDS_DIR, exist_ok=True)
np.save(f"{PREDS_DIR}/y_pred_lora.npy", y_pred_lora)
np.save(f"{PREDS_DIR}/y_pred_ensemble.npy", y_pred_ensemble)
print("Predicciones guardadas en Drive")


## 9. Resultados comparativos


In [ ]:
model_names = [r['model_name'] for r in results]
f1_scores = [r['f1_macro'] for r in results]
class_labels = ['Negativo', 'Neutro', 'Positivo']

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
bars = plt.barh(range(len(results)), f1_scores, color=colors[:len(results)])
plt.yticks(range(len(results)), model_names)
plt.xlabel('F1-macro')
plt.title('Comparacion de Modelos - F1-macro')
plt.xlim(0, 1)
for bar, val in zip(bars, f1_scores):
    plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=10)

plt.subplot(1, 2, 2)
x = np.arange(len(class_labels))
width = 0.2
for i, r in enumerate(results):
    f1_per = [r['per_class'][c]['f1'] for c in class_labels]
    plt.bar(x + i*width, f1_per, width, label=r['model_name'], color=colors[i])
plt.xticks(x + width * 1.5, class_labels)
plt.ylabel('F1-score')
plt.title('F1 por clase')
plt.legend(loc='best', fontsize=8)
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("Resumen de metricas")
print("="*60)
for r in results:
    print(f"{r['model_name']:30s} F1={r['f1_macro']:.4f}  Acc={r['accuracy']:.4f}  T={r['training_time_seconds']:.0f}s")


## 10. MLflow Tracking


In [ ]:
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "https://humorous-trusting-domelike.ngrok-free.dev")
import requests
try:
    r = requests.get(f"{MLFLOW_TRACKING_URI}/api/2.0/mlflow/experiments/list", timeout=5)
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
except Exception:
    mlflow.set_tracking_uri(f"sqlite:///{DRIVE_BASE}/mlflow_fallback.db")

mlflow.set_experiment("distilbert_improved")

for r in results:
    with mlflow.start_run(run_name=r['model_name']):
        mlflow.log_params({'model_name': r['model_name']})
        mlflow.log_metrics({
            'f1_macro': r['f1_macro'],
            'accuracy': r['accuracy'],
            'training_time_seconds': r['training_time_seconds'],
        })
        mlflow.log_dict(r['confusion_matrix'], f"{r['model_name']}_confusion_matrix.json")

print("MLflow tracking completado")


## 11. Exportar métricas a JSON


In [ ]:
report_path = f"{REPORTS_DIR}/metrics_distilbert_improved.json"
with open(report_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Exportado: {report_path}")


In [ ]:
# Liberar memoria
del model_lora, trainer_lora, model_cls, tokenizer
del X_train_emb, X_val_emb, X_test_emb, eng_train, eng_val, eng_test
del X_train, X_val, X_test, y_train, y_val, y_test
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\nF3-D completado. Todos los modelos entrenados.")
